# Traffic Demand — Hard Stacked Ensemble (5 model families)

Out-of-fold **stacking** of LightGBM + XGBoost + CatBoost + HistGradientBoosting + ExtraTrees, combined by a Ridge meta-learner. Features = the geohash×time target-encoding set + lat/lon + geo-cluster + cyclical time + day-48 profile (the set that reached ~90 on the leaderboard), with the boosters using `geohash` etc. as **native categoricals**.

**Expectation (honest):** diverse models on this data tend to converge near **R² ≈ 0.90–0.91**, because the variance separating the forecast day from the training day isn't fully captured by the features. Stacking may add ~1 point. Run it and compare to your current best.

## Setup

In [ ]:
!pip -q install lightgbm xgboost catboost 2>/dev/null; print('ready')

## Upload data

In [ ]:
from google.colab import files
print('Upload train.csv, test.csv, sample_submission.csv:')
up=files.upload()

## Features (geohash×time encodings + profile + native categoricals)

In [ ]:
import numpy as np, pandas as pd, warnings
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
warnings.filterwarnings('ignore')

# Core feature engineering (the 90.3 feature set, calibration removed) + heavy-model ensemble.
import numpy as np, pandas as pd, warnings
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
warnings.filterwarnings("ignore")
SEED=42

_B32="0123456789bcdefghjkmnpqrstuvwxyz"; _DEC={c:i for i,c in enumerate(_B32)}
def gh_decode(gh):
    if not isinstance(gh,str) or not gh: return (np.nan,np.nan)
    lat,lon,is_lon=[-90.,90.],[-180.,180.],True
    for ch in gh.lower():
        cd=_DEC.get(ch)
        if cd is None: continue
        for mask in (16,8,4,2,1):
            if is_lon: mid=(lon[0]+lon[1])/2; lon[0 if cd&mask else 1]=mid
            else:      mid=(lat[0]+lat[1])/2; lat[0 if cd&mask else 1]=mid
            is_lon=not is_lon
    return ((lat[0]+lat[1])/2,(lon[0]+lon[1])/2)
def mod_of(t): h,m=str(t).split(":"); return int(h)*60+int(m)

def kfold_te(ktr,kte,y,folds,sm=10.0):
    gm=y.mean(); oof=pd.Series(np.nan,index=ktr.index)
    for tr,va in folds:
        s=pd.DataFrame({"k":ktr.iloc[tr],"y":y.iloc[tr]}).groupby("k")["y"].agg(["mean","count"])
        smv=(s["mean"]*s["count"]+gm*sm)/(s["count"]+sm); oof.iloc[va]=ktr.iloc[va].map(smv).values
    oof=oof.fillna(gm)
    f=pd.DataFrame({"k":ktr,"y":y}).groupby("k")["y"].agg(["mean","count"])
    smf=(f["mean"]*f["count"]+gm*sm)/(f["count"]+sm)
    return oof.values, kte.map(smf).fillna(gm).values
def mkkey(df,cols):
    s=df[cols[0]].astype(str)
    for c in cols[1:]: s=s+"|"+df[c].astype(str)
    return s

def build(train,test,y,folds):
    def base(df):
        o=pd.DataFrame(index=df.index); m=df["timestamp"].map(mod_of)
        o["mod"]=m; o["hour"]=m//60; o["minute"]=m%60
        a=2*np.pi*m/1440.; o["mod_sin"],o["mod_cos"]=np.sin(a),np.cos(a)
        o["day"]=pd.to_numeric(df["day"],errors="coerce")
        gh=df["geohash"].astype(str); c={g:gh_decode(g) for g in gh.dropna().unique()}
        o["gh_lat"]=gh.map(lambda g:c[g][0]); o["gh_lon"]=gh.map(lambda g:c[g][1])
        for p in (4,5): o[f"gh_pre{p}"]=gh.str.slice(0,p).astype("category").cat.codes
        o["gh_code"]=gh.astype("category").cat.codes
        o["NumberofLanes"]=pd.to_numeric(df["NumberofLanes"],errors="coerce")
        o["Temperature"]=pd.to_numeric(df["Temperature"],errors="coerce")
        o["LargeVehicles"]=df["LargeVehicles"].astype(str).str.strip().str.lower().map({"allowed":1,"not allowed":0}).fillna(-1)
        o["Landmarks"]=df["Landmarks"].astype(str).str.strip().str.lower().map({"yes":1,"no":0}).fillna(-1)
        o["RoadType"]=df["RoadType"].astype("category").cat.codes
        o["Weather"]=df["Weather"].astype("category").cat.codes
        return o,m
    Xtr,mtr=base(train); Xte,mte=base(test)
    Xtr=Xtr.reset_index(drop=True); Xte=Xte.reset_index(drop=True); yr=y.reset_index(drop=True)
    rt,re=train.copy(),test.copy(); rt["mod"],re["mod"]=mtr.values,mte.values; rt["hour"],re["hour"]=mtr.values//60,mte.values//60
    for k in (4,5): rt[f"gh{k}"]=train.geohash.astype(str).str.slice(0,k); re[f"gh{k}"]=test.geohash.astype(str).str.slice(0,k)
    specs=[["geohash"],["geohash","hour"],["geohash","mod"],["gh4"],["gh4","hour"],["RoadType"],["Weather"],
           ["geohash","RoadType"],["gh5","hour"],["RoadType","hour"],["Weather","hour"]]
    for cols in specs:
        otr,ote=kfold_te(mkkey(rt,cols).reset_index(drop=True),mkkey(re,cols).reset_index(drop=True),yr,folds)
        Xtr["te_"+"_".join(cols)]=otr; Xte["te_"+"_".join(cols)]=ote
    d49=train[train.day==49]; gma=train.groupby("geohash")["demand"].mean(); gm=y.mean()
    rec=d49.groupby("geohash")["demand"].mean()
    rf=lambda df: df.geohash.map(rec).fillna(df.geohash.map(gma)).fillna(gm).values
    Xtr["recent_gh"]=rf(train); Xte["recent_gh"]=rf(test)
    d48=train[train.day==48].copy(); d48["mod"]=d48.timestamp.map(mod_of)
    prof=d48.groupby(["geohash","mod"])["demand"].mean(); piv=prof.unstack("mod").sort_index(axis=1); pT=piv.T.sort_index()
    rs=pT.rolling(5,center=True,min_periods=1).sum(); rc=pT.notna().rolling(5,center=True,min_periods=1).sum()
    sx=((rs-pT.fillna(0))/(rc-pT.notna()).replace(0,np.nan)).T.stack().to_dict()
    si=pT.rolling(5,center=True,min_periods=1).mean().T.stack().to_dict(); pd_=prof.to_dict()
    lk=lambda d,gs,ms: np.array([d.get((g,m),np.nan) for g,m in zip(gs,ms)])
    gt,ge=train.geohash.values,test.geohash.values
    Xtr["d48_smooth"]=lk(sx,gt,mtr.values); Xte["d48_smooth"]=lk(si,ge,mte.values)
    for off,nm in [(-15,"prev"),(15,"next"),(-30,"prev2"),(30,"next2")]:
        Xtr[f"d48_{nm}"]=lk(pd_,gt,mtr.values+off); Xte[f"d48_{nm}"]=lk(pd_,ge,mte.values+off)
    Xtr["d48_trend"]=Xtr["d48_next"]-Xtr["d48_prev"]; Xte["d48_trend"]=Xte["d48_next"]-Xte["d48_prev"]
    # raw categoricals for native-categorical models (CatBoost/LightGBM/XGBoost)
    cat_cols=[]
    for c in ["geohash","gh4","gh5","RoadType","Weather","LargeVehicles","Landmarks"]:
        src_tr = (train.geohash.astype(str).str.slice(0,4) if c=="gh4" else
                  train.geohash.astype(str).str.slice(0,5) if c=="gh5" else train[c].astype(str))
        src_te = (test.geohash.astype(str).str.slice(0,4) if c=="gh4" else
                  test.geohash.astype(str).str.slice(0,5) if c=="gh5" else test[c].astype(str))
        Xtr["cat_"+c]=src_tr.values; Xte["cat_"+c]=src_te.values; cat_cols.append("cat_"+c)
    cols=sorted(set(Xtr.columns)&set(Xte.columns))
    return Xtr[cols], Xte[cols], cat_cols

## Train 5 model families with out-of-fold predictions, then Ridge-stack

In [ ]:
tr=pd.read_csv('train.csv'); te=pd.read_csv('test.csv'); samp=pd.read_csv('sample_submission.csv')
y=pd.to_numeric(tr['demand'],errors='coerce'); lo,hi=0.0,y.max()
folds=list(KFold(5,shuffle=True,random_state=42).split(tr))
Xtr,Xte,cat_cols=build(tr,te,y,folds)
num=[c for c in Xtr.columns if not c.startswith('cat_')]
print('features:',Xtr.shape[1],'| categorical:',len(cat_cols))

def oof_run(name, fit_fn, predict_fn):
    o=np.zeros(len(Xtr)); p=np.zeros(len(Xte))
    for tri,vai in folds:
        model=fit_fn(Xtr.iloc[tri], np.log1p(y.iloc[tri]), Xtr.iloc[vai], np.log1p(y.iloc[vai]))
        o[vai]=np.clip(np.expm1(predict_fn(model,Xtr.iloc[vai])),lo,hi)
        p+=np.clip(np.expm1(predict_fn(model,Xte)),lo,hi)/len(folds)
    print(f'{name:8s} OOF R2={r2_score(y,o):.4f}')
    return o,p

OOF={}; PRED={}
# LightGBM (native categoricals)
try:
    import lightgbm as lgb
    Acat=Xtr.copy(); Bcat=Xte.copy()
    for c in cat_cols: Acat[c]=Acat[c].astype('category'); Bcat[c]=Bcat[c].astype('category')
    def f_lgb(a,ya,v,yv):
        m=lgb.LGBMRegressor(n_estimators=4000,learning_rate=0.02,num_leaves=160,subsample=0.8,subsample_freq=1,colsample_bytree=0.8,reg_lambda=2.0,min_child_samples=30,random_state=42,n_jobs=-1,verbose=-1)
        m.fit(a.loc[:,Acat.columns], ya, eval_set=[(v.loc[:,Acat.columns],yv)], categorical_feature=cat_cols, callbacks=[lgb.early_stopping(150,verbose=False)]); return m
    # use category-typed frames
    def fit_lgb(tri_a,tri_y,va_a,va_y): return f_lgb(tri_a,tri_y,va_a,va_y)
    o=np.zeros(len(Xtr)); p=np.zeros(len(Xte))
    for tri,vai in folds:
        m=lgb.LGBMRegressor(n_estimators=4000,learning_rate=0.02,num_leaves=160,subsample=0.8,subsample_freq=1,colsample_bytree=0.8,reg_lambda=2.0,min_child_samples=30,random_state=42,n_jobs=-1,verbose=-1)
        m.fit(Acat.iloc[tri],np.log1p(y.iloc[tri]),eval_set=[(Acat.iloc[vai],np.log1p(y.iloc[vai]))],categorical_feature=cat_cols,callbacks=[lgb.early_stopping(150,verbose=False)])
        o[vai]=np.clip(np.expm1(m.predict(Acat.iloc[vai])),lo,hi); p+=np.clip(np.expm1(m.predict(Bcat)),lo,hi)/len(folds)
    OOF['lgb'],PRED['lgb']=o,p; print(f'lgb      OOF R2={r2_score(y,o):.4f}')
except Exception as e: print('lgb skipped:',e)

# XGBoost (categorical)
try:
    import xgboost as xgb
    Ax=Xtr.copy(); Bx=Xte.copy()
    for c in cat_cols: Ax[c]=Ax[c].astype('category'); Bx[c]=Bx[c].astype('category')
    o=np.zeros(len(Xtr)); p=np.zeros(len(Xte))
    for tri,vai in folds:
        m=xgb.XGBRegressor(n_estimators=4000,learning_rate=0.02,max_depth=9,subsample=0.8,colsample_bytree=0.8,reg_lambda=2.0,tree_method='hist',enable_categorical=True,random_state=42,n_jobs=-1,early_stopping_rounds=150,eval_metric='rmse')
        m.fit(Ax.iloc[tri],np.log1p(y.iloc[tri]),eval_set=[(Ax.iloc[vai],np.log1p(y.iloc[vai]))],verbose=False)
        o[vai]=np.clip(np.expm1(m.predict(Ax.iloc[vai])),lo,hi); p+=np.clip(np.expm1(m.predict(Bx)),lo,hi)/len(folds)
    OOF['xgb'],PRED['xgb']=o,p; print(f'xgb      OOF R2={r2_score(y,o):.4f}')
except Exception as e: print('xgb skipped:',e)

# CatBoost (native categoricals)
try:
    from catboost import CatBoostRegressor
    Ac=Xtr.copy(); Bc=Xte.copy()
    for c in cat_cols: Ac[c]=Ac[c].astype(str); Bc[c]=Bc[c].astype(str)
    idx=[Ac.columns.get_loc(c) for c in cat_cols]
    o=np.zeros(len(Xtr)); p=np.zeros(len(Xte))
    for tri,vai in folds:
        m=CatBoostRegressor(iterations=5000,learning_rate=0.03,depth=8,l2_leaf_reg=3.0,loss_function='RMSE',random_seed=42,verbose=0)
        m.fit(Ac.iloc[tri],np.log1p(y.iloc[tri]),cat_features=idx,eval_set=(Ac.iloc[vai],np.log1p(y.iloc[vai])),use_best_model=True)
        o[vai]=np.clip(np.expm1(m.predict(Ac.iloc[vai])),lo,hi); p+=np.clip(np.expm1(m.predict(Bc)),lo,hi)/len(folds)
    OOF['cat'],PRED['cat']=o,p; print(f'cat      OOF R2={r2_score(y,o):.4f}')
except Exception as e: print('cat skipped:',e)

# HGBR + ExtraTrees (numeric)
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
An=Xtr[num].fillna(0); Bn=Xte[num].fillna(0)
for nm,mk in [('hgbr',lambda:HistGradientBoostingRegressor(max_iter=3000,learning_rate=0.03,max_leaf_nodes=160,min_samples_leaf=30,l2_regularization=1.0,early_stopping=True,random_state=42)),
              ('et',  lambda:ExtraTreesRegressor(n_estimators=300,max_features=0.5,min_samples_leaf=8,n_jobs=-1,random_state=42))]:
    o=np.zeros(len(Xtr)); p=np.zeros(len(Xte))
    for tri,vai in folds:
        m=mk(); m.fit(An.iloc[tri],np.log1p(y.iloc[tri]))
        o[vai]=np.clip(np.expm1(m.predict(An.iloc[vai])),lo,hi); p+=np.clip(np.expm1(m.predict(Bn)),lo,hi)/len(folds)
    OOF[nm],PRED[nm]=o,p; print(f'{nm:8s} OOF R2={r2_score(y,o):.4f}')

keys=list(OOF.keys()); print('\nmodels:',keys)
SX=np.column_stack([OOF[k] for k in keys]); SXt=np.column_stack([PRED[k] for k in keys])
meta=Ridge(alpha=1.0,positive=True).fit(SX,y)
stack=np.clip(meta.predict(SXt),lo,hi)
avg=np.clip(np.mean([PRED[k] for k in keys],axis=0),lo,hi)
print('AVG   OOF R2=%.4f'%r2_score(y,np.mean([OOF[k] for k in keys],axis=0)))
print('STACK OOF R2=%.4f  weights=%s'%(r2_score(y,np.clip(meta.predict(SX),lo,hi)), dict(zip(keys,np.round(meta.coef_,3)))))

sub=pd.DataFrame({'Index':te['Index'].values,'demand':stack})[list(samp.columns)]
assert sub.shape==(len(te),2) and sub['demand'].notna().all()
sub.to_csv('submission.csv',index=False)
print('wrote submission.csv (stacked) mean',round(stack.mean(),4))
files.download('submission.csv'); sub.head()